## **Setup**

In [1]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import gdown
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
import lightgbm as lgbm

import tensorflow as tf
import tensorflow_recommenders as tfrs
from tensorflow.keras.layers import StringLookup, TextVectorization, Embedding, GRU, Dense
from tensorflow.keras import layers

2025-11-18 14:35:39.840286: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-18 14:35:40.186524: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-18 14:35:42.493604: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## **Data preparation**

In [2]:
# Download data
file_id = "1GffOYmcAMP17oi2BwC7Dp4l5F7rEjHRr" 
url = f"https://drive.google.com/uc?id={file_id}"
output = "mind_large.zip"
is_downloaded = False

for file in os.listdir("."):
    if file.startswith("mind_large"):
        print("mind_large is already downloaded.")
        is_downloaded = True
        break
        
if is_downloaded == False:
    print("Downloading mind_large.zip ...")
    gdown.download(url, output, quiet=False)
    print("Download complete!")

    # Extract compressed file
    with zipfile.ZipFile("mind_large.zip", "r") as z:
        z.extractall(".")

mind_large is already downloaded.


In [3]:
# Sets
sets = ["train", "dev", "test"]

# News
news_header = ["id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]
news = {}
for _set in sets:
    news[_set] = pd.read_csv(f"mind_large/news_{_set}.tsv", names=news_header, sep="\t")
all_news_df = pd.concat([news["train"], news["dev"], news["test"]], ignore_index=True)    

# Impressions
behaviors_header = ["impression_id", "user_id", "time", "history", "impressions"]
behaviors = {}
for _set in sets:
    behaviors[_set] = pd.read_csv(f"mind_large/behaviors_{_set}.tsv", names=behaviors_header, sep="\t")
all_behaviors_df = pd.concat([behaviors["train"], behaviors["dev"], behaviors["test"]], ignore_index=True)    

In [4]:
all_news_df = all_news_df.drop(columns=["url", "title_entities", "abstract_entities"])
all_news_df.drop_duplicates(inplace=True)
all_news_df["abstract"] = all_news_df["abstract"].fillna(all_news_df["title"])

In [5]:
all_behaviors_df["history"] = all_behaviors_df["history"].fillna("")

## **Filtering data (to keep relevant rows)**

In [6]:
len(all_behaviors_df)

4979946

In [7]:
behaviors_df = all_behaviors_df[
    (all_behaviors_df["impressions"].str.contains("-1")) & # Keep only impressions with at least one click
    (all_behaviors_df["history"].str.len() > 0) & # Keep users with AT LEAST ONE item in their history
    (all_behaviors_df["impressions"].str.len() >= 10) # Keep only impressions with at least 2 items shown
]

len(behaviors_df)

2551884

In [8]:
# Sampling (my computer is not that performant :))
behaviors_df = behaviors_df.sample(n=20_000)

## **Building the user-news interaction dataset**

In [9]:
# Function to split the impressions and clicks into two separate lists
def process_impression(impression_list):
    clicked, non_clicked = [], []
    if impression_list != "":
        list_of_strings = impression_list.split()
        clicked = [x.split("-")[0] for x in list_of_strings if x.split("-")[1] == "1"]
        non_clicked = [x.split("-")[0] for x in list_of_strings if x.split("-")[1] == "0"]
    return clicked, non_clicked

In [10]:
# Separate views from clicks
behaviors_df[["clicked", "non_clicked"]] = behaviors_df["impressions"].apply(
    lambda x: pd.Series(process_impression(x))
)
# Split history
behaviors_df["history"] = behaviors_df["history"].apply(lambda x: x.split())

behaviors_df.head()

,impression_id,user_id,time,history,impressions,clicked,non_clicked
258565,258566,U549795,11/12/2019 10:37:50 AM,"[N7154, N128965, N23920, N47655, N58653, N1191...",N3075-1 N66780-0 N96373-0 N38902-0 N82719-0 N1...,[N3075],"[N66780, N96373, N38902, N82719, N109642, N302..."
531102,531103,U163654,11/14/2019 8:42:29 AM,"[N3948, N45124, N45124, N16356, N63630]",N29544-0 N58641-0 N105407-0 N76665-1,[N76665],"[N29544, N58641, N105407]"
458096,458097,U80319,11/10/2019 9:12:28 AM,"[N48412, N81289, N2076, N83979, N63295, N73122...",N31174-1 N2591-0 N78206-0 N46652-0 N40282-0 N5...,[N31174],"[N2591, N78206, N46652, N40282, N54460, N58592..."
880946,880947,U421702,11/13/2019 8:42:37 AM,"[N27258, N124989, N12959, N63560, N16078, N197...",N12453-0 N20964-0 N120696-0 N99964-0 N72609-0 ...,[N96373],"[N12453, N20964, N120696, N99964, N72609, N967..."
829479,829480,U481302,11/14/2019 1:58:23 PM,"[N82348, N96616, N108752, N79529, N34366, N100...",N85089-0 N20250-0 N76610-0 N29441-0 N67937-1 N...,[N67937],"[N85089, N20250, N76610, N29441, N45410, N6544..."


In [11]:
%%time

click_data = []
for _, row in behaviors_df.iterrows():
    history = row["history"]
    clicked_news, non_clicked_news = row["clicked"], row["non_clicked"]
    
    for news_id in clicked_news:
        click_data.append({
            "history": history,
            "candidate_news_id": news_id,
            "label": 1
        })
        
    # We can also sample non-clicked news for harder negatives
    for news_id in non_clicked_news:
         click_data.append({
            "history": history,
            "candidate_news_id": news_id,
            "label": 0
        })

# Create a DataFrame from the exploded data
training_df = pd.DataFrame(click_data)
training_df.head()

CPU times: user 1.99 s, sys: 70.6 ms, total: 2.06 s
Wall time: 2.06 s


,history,candidate_news_id,label
0,"[N7154, N128965, N23920, N47655, N58653, N1191...",N3075,1
1,"[N7154, N128965, N23920, N47655, N58653, N1191...",N66780,0
2,"[N7154, N128965, N23920, N47655, N58653, N1191...",N96373,0
3,"[N7154, N128965, N23920, N47655, N58653, N1191...",N38902,0
4,"[N7154, N128965, N23920, N47655, N58653, N1191...",N82719,0


In [12]:
# For Retrieval (Two-Tower): We only need positive interactions (label=1)
retrieval_df = training_df[training_df["label"] == 1].copy()

# This will be used to join features to the candidate_news_id
news_features_df = all_news_df[["id", "category", "title"]].copy()
news_features_df = news_features_df.rename(columns={"id": "candidate_news_id"})

# Merge retrieval_df with all_news_df
retrieval_df_merged = retrieval_df.merge(
    news_features_df,
    on="candidate_news_id",
    how="left"
)

# We now use the merged DataFrame which contains all the required features.
# We also use the keys that compute_loss expects ("news_id", "category", "title")
retrieval_ds = tf.data.Dataset.from_tensor_slices({
    "history": tf.ragged.constant(retrieval_df_merged["history"].values),
    "news_id": tf.constant(retrieval_df_merged["candidate_news_id"].values),
    "category": tf.constant(retrieval_df_merged["category"].values),
    "title": tf.constant(retrieval_df_merged["title"].values)
})

print(f"Created {len(retrieval_df_merged)} positive pairs for retrieval training.")

Created 30437 positive pairs for retrieval training.


E0000 00:00:1763472997.086101   62098 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1763472997.098470   62098 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2025-11-18 14:36:37.101577: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 29761920 exceeds 10% of free system memory.


## **Stage 1: Retrieval (Two-Tower Model)**

**Defining the hyperparameters for the training**

In [13]:
EMBEDDING_DIM = 64 # Output embeddings dimension for both User and News tower
MAX_HISTORY_LENGTH = 30 # Max number of articles to look at in user history
MAX_TOKENS = 20000 # Max vocab size for titles
TITLE_VECTORIZATION_DIM = 100

**Building the News Tower**

This model turns a NewsID into an embedding

In [14]:
class NewsModel(tf.keras.Model):
    def __init__(self, all_news_ids, all_categories, **kwargs):
        super().__init__(**kwargs)
        
        self.all_news_ids = all_news_ids
        self.all_categories = all_categories
        
        # News ID embedding model
        self.news_id_lookup = StringLookup(vocabulary=self.all_news_ids, mask_token=None)
        self.news_id_embedding_model = Embedding(input_dim=len(self.all_news_ids) + 1, output_dim=EMBEDDING_DIM)
        
        # Category embedding model
        self.category_lookup = StringLookup(vocabulary=self.all_categories, mask_token=None)
        self.category_embedding_model = Embedding(input_dim=len(self.all_categories) + 1, output_dim=EMBEDDING_DIM)
        
        # Title vectorizer
        self.title_vectorizer = TextVectorization(
            max_tokens=MAX_TOKENS,
            output_mode="int",
            output_sequence_length=TITLE_VECTORIZATION_DIM
        )
                
        # Title embedding model
        self.title_embedding_model = tf.keras.Sequential([
            Embedding(input_dim=MAX_TOKENS, output_dim=EMBEDDING_DIM),
            tf.keras.layers.GlobalAveragePooling1D() # Average word embeddings
        ])
        
        # Final Dense Layer
        self.dense = Dense(EMBEDDING_DIM)
        
        # Store news data for quick lookup
        self.news_data = tf.data.Dataset.from_tensor_slices({
            "news_id": all_news_df["id"].values,
            "category": all_news_df["category"].values,
            "title": all_news_df["title"].values
        }).batch(128)

            
    
    def call(self, inputs):
        
        # Get embedding for each feature
        news_id_embedding = self.news_id_embedding_model(self.news_id_lookup(inputs["news_id"]))
        category_embedding = self.category_embedding_model(self.category_lookup(inputs["category"]))
        title_embedding = self.title_embedding_model(self.title_vectorizer(inputs["title"]))
        
        # Combine them
        combined_embeddings = tf.concat([news_id_embedding, category_embedding, title_embedding], axis=1)
        
        # Pass through the final dense layer to get a single 64-dim vector
        return self.dense(combined_embeddings)
    
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "all_news_ids": self.all_news_ids,
            "all_categories": self.all_categories,
        })
        return config

**Build the User (Session) Tower**

This model turns a user's click history into an embedding

In [15]:
class UserModel(tf.keras.Model):
    def __init__(self, news_id_embedding_model, all_news_ids, **kwargs):
        super().__init__(**kwargs)
        # Use the same embedding layer as the news model        
        self.news_id_embedding_model = news_id_embedding_model
        self.all_news_ids = all_news_ids
        self.news_id_lookup = StringLookup(vocabulary=self.all_news_ids, mask_token=None)
        
        # We use a GRU to process the sequence of clicked news
        self.gru = GRU(EMBEDDING_DIM)
        

    def call(self, history):
        history = history[:, -MAX_HISTORY_LENGTH:]
        history_int = self.news_id_lookup(history)
        
        # Get embeddings for each news ID in the history
        history_embeddings = self.news_id_embedding_model(history_int)
        
        # Output
        return self.gru(history_embeddings)
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "all_news_ids": self.all_news_ids,
        })
        return config

**Combining the two towers**

In [16]:
class MINDRetrievalModel(tfrs.Model):
    def __init__(self, user_model, news_model, candidate_dataset):
        super().__init__()
        self.user_model = user_model
        self.news_model = news_model
        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=candidate_dataset.map(self.news_model),
                    ks=[20]
            )
        )    
    
    
    def compute_loss(self, features, training=False):
                
        user_embeddings = self.user_model(features["history"])
        
        # Create the dictionary for the news model
        news_features = {
            "news_id": features["news_id"],
            "category": features["category"],
            "title": features["title"]
        }
        positive_news_embeddings = self.news_model(news_features)
        
        return self.task(user_embeddings, positive_news_embeddings)

**Training the model**

In [17]:
# Define datasets
train_ds = retrieval_ds.take(18_000)
val_ds = retrieval_ds.skip(18_000).take(1_000)
test_ds = retrieval_ds.skip(19_000)

# Batch data
train_ds_batched = train_ds.shuffle(18_000).batch(128).cache()
val_ds_batched   = val_ds.batch(128).cache()
test_ds_batched  = test_ds.batch(128).cache()

# Use for computing metrics in the Two-Tower
candidate_dataset = tf.data.Dataset.from_tensor_slices({
    "news_id": all_news_df["id"].values,
    "category": all_news_df["category"].values,
    "title": all_news_df["title"].values
})

In [18]:
# Vocabulary for categorical features (id + category)
all_news_ids = all_news_df["id"].unique()
all_categories = all_news_df["category"].unique()
news_ids_list = all_news_ids.tolist() if hasattr(all_news_ids, "tolist") else all_news_ids
categories_list = all_categories.tolist() if hasattr(all_categories, "tolist") else all_categories

# News model
news_model = NewsModel(all_news_ids=news_ids_list, all_categories=categories_list)
news_model.title_vectorizer.adapt(all_news_df["title"])

# User model
user_model = UserModel(news_id_embedding_model=news_model.news_id_embedding_model,
                       all_news_ids=news_ids_list)

# Retrieval task
sampled_candidate_dataset = candidate_dataset.batch(128).take(200) # We just take a little part to avoid computation issues
retrieval_model = MINDRetrievalModel(user_model=user_model, news_model=news_model, candidate_dataset=sampled_candidate_dataset)
retrieval_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))

2025-11-18 14:36:41.894418: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 33377280 exceeds 10% of free system memory.
2025-11-18 14:36:41.901821: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 33377280 exceeds 10% of free system memory.
2025-11-18 14:36:41.910740: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 33377280 exceeds 10% of free system memory.


In [19]:
# Train for a few epochs
history = retrieval_model.fit(train_ds_batched,
                              validation_data=val_ds_batched,
                              epochs=3)

Epoch 1/3


2025-11-18 14:36:44.817677: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 33377280 exceeds 10% of free system memory.


141/141 [==============================] - 75s 499ms/step - factorized_top_k/top_20_categorical_accuracy: 0.0103 - loss: 615.2867 - regularization_loss: 0.0000e+00 - total_loss: 615.2867 - val_factorized_top_k/top_20_categorical_accuracy: 0.0160 - val_loss: 477.3113 - val_regularization_loss: 0.0000e+00 - val_total_loss: 477.3113
Epoch 2/3
141/141 [==============================] - 71s 503ms/step - factorized_top_k/top_20_categorical_accuracy: 0.0739 - loss: 595.9586 - regularization_loss: 0.0000e+00 - total_loss: 595.9586 - val_factorized_top_k/top_20_categorical_accuracy: 0.0620 - val_loss: 471.8202 - val_regularization_loss: 0.0000e+00 - val_total_loss: 471.8202
Epoch 3/3
141/141 [==============================] - 70s 497ms/step - factorized_top_k/top_20_categorical_accuracy: 0.1381 - loss: 568.7845 - regularization_loss: 0.0000e+00 - total_loss: 568.7845 - val_factorized_top_k/top_20_categorical_accuracy: 0.0950 - val_loss: 473.1600 - val_regularization_loss: 0.0000e+00 - val_total

In [20]:
# Evaluate the model
metrics = retrieval_model.evaluate(test_ds_batched, return_dict=True)
metrics

90/90 [==============================] - 45s 502ms/step - factorized_top_k/top_20_categorical_accuracy: 0.0808 - loss: 612.4059 - regularization_loss: 0.0000e+00 - total_loss: 612.4059


{'factorized_top_k/top_20_categorical_accuracy': 0.08079041540622711,
 'loss': 169.62440490722656,
 'regularization_loss': 0,
 'total_loss': 169.62440490722656}

In [21]:
# Save the models for inference 

# We need the user tower to get user embeddings and )
user_model.save("models/user_retrieval_model", save_format="tf")

# We need the news tower to build the candidate index
news_model.save("models/news_retrieval_model", save_format="tf")

INFO:tensorflow:Assets written to: models/user_retrieval_model/assets


INFO:tensorflow:Assets written to: models/user_retrieval_model/assets


INFO:tensorflow:Assets written to: models/news_retrieval_model/assets


INFO:tensorflow:Assets written to: models/news_retrieval_model/assets


## **Stage 2: Ranking (GDBT Model)**

In [22]:
# Load the trained embedding models

user_model = tf.keras.models.load_model("models/user_retrieval_model")
news_model = tf.keras.models.load_model("models/news_retrieval_model")

In [23]:
# Create a full news embedding lookup (dictionary)

all_news_embeddings = {}
for news_id_batch in candidate_dataset.batch(512):
    embeddings_batch = news_model(news_id_batch)
    news_ids = news_id_batch["news_id"].numpy()
    for news_id, embedding in zip(news_ids, embeddings_batch.numpy()):
        all_news_embeddings[news_id.decode("utf-8")] = embedding

2025-11-18 14:41:24.208336: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [24]:
# Create user embedding cache
user_embeddings_cache = {}

# Helper function to get features
def get_features(row):
    features = {}
    
    # Get user history
    user_history = row["history"]
    history_key = tuple(user_history) # Use tuple as dict key
    
    # Get user embedding
    if history_key in user_embeddings_cache:
        user_emb = user_embeddings_cache[history_key]
    else:
        user_emb = user_model(tf.ragged.constant([user_history])).numpy()[0]
        user_embeddings_cache[history_key] = user_emb
        
    # Get candidate news embedding
    candidate_id = row["candidate_news_id"]
    candidate_emb = all_news_embeddings.get(candidate_id, np.zeros(EMBEDDING_DIM))
    
    # Get some features for the model
    
    # 1. Retrieval Score (dot product)
    features["retrieval_dot_product"] = np.dot(user_emb, candidate_emb)
    
    # 2. User Features
    features["history_length"] = len(user_history)
    
    # 3. Candidate News Features (lookup from news_df)
    news_info = all_news_df.loc[all_news_df["id"] == candidate_id].iloc[0]
    features["category"] = news_model.category_lookup(news_info["category"]).numpy()
    
    # 4. Cross Features
    # How many times this category appeared in history ?
    history_categories = all_news_df[all_news_df["id"].isin(user_history)]["category"].values
    features["category_in_history_count"] = np.sum(history_categories == news_info["category"])
    
    return features

In [25]:
# Training data

ranking_sample_df = training_df.sample(n=2_000, random_state=42)

features_list = ranking_sample_df.apply(get_features, axis=1)
X = pd.DataFrame.from_records(features_list.tolist())
y = ranking_sample_df["label"]

# Convert categorical features for LightGBM
X["category"] = X["category"].astype("category")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [26]:
X_train.head()

,retrieval_dot_product,history_length,category,category_in_history_count
968,-2.262980,52,2,22
240,1.662386,28,4,7
819,0.035772,32,14,0
692,-0.180006,31,2,20
420,-2.308912,39,2,5


In [27]:
# Train the model

lgbm_train = lgbm.Dataset(X_train, y_train, categorical_feature=["category"])
lgbm_eval = lgbm.Dataset(X_test, y_test, reference=lgbm_train, categorical_feature=["category"])

params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "verbose": -1
}

lgbm_model = lgbm.train(
    params,
    lgbm_train,
    num_boost_round=500,
    callbacks=[lgbm.early_stopping(10, verbose=False)],
    valid_sets=[lgbm_eval]
)

In [28]:
# Save the model

lgbm_model.save_model("models/ranking_model.txt")

## **Resources**

* https://www.tensorflow.org/recommenders/examples/basic_retrieval
* https://www.kaggle.com/code/jacobwelander/mind-recommender-from-scratch-2023
* https://www.kaggle.com/code/kanruwang/tensorflow-recommender-two-tower-multitask